# Names, Types & Generics — Polyglot Reference

How names refer to values, what types those names carry, and how generics let one declaration cover many types.

**Languages:** Java · Scala · Kotlin · JavaScript · TypeScript · Python

This notebook covers four mental clusters:

1. **Names & scoping** — where a binding is visible, how shadowing and closures behave
2. **Type annotations & inference** — what types look like in source, when they're inferred
3. **Generics** — type parameters, bounds, variance
4. **Modules & imports** — file/namespace organization, how names cross file boundaries

For value semantics (`val`/`var`/immutability), see `01-values.ipynb`. Concurrency-related scope concerns (async-local, thread-local) are deferred to per-language repos.

## Names & Scoping

### Where names live — the lookup chain

When you reference a name, the runtime walks outward from the innermost scope until it finds a binding (or fails). The chain differs across the three families: JVM-based, JavaScript/TypeScript, and Python.

![Three scope chains: JVM family, JavaScript/TypeScript, Python LEGB](https://raw.githubusercontent.com/schemabotview/polyglot/main/img/names-scope-chain.svg)

- **Java / Scala / Kotlin** — `block → method → class → file/package → stdlib + imports`. Every name in a class is implicitly accessible from any method on that class. Imports are resolved at the file level.
- **JavaScript / TypeScript** — `block (let/const) → function → module → globalThis`. The historical `var` keyword is *function*-scoped and **hoists** to the top of the enclosing function — a notorious source of bugs that `let` and `const` fix.
- **Python — LEGB** — `Local → Enclosing → Global (module) → Built-in`. Notice what's missing: **classes are not a scope level for lookup**. Inside a method, `foo` does *not* see a class-level `foo` — you must write `self.foo` or `ClassName.foo`. A frequent surprise for JVM programmers.

### Sub-features

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| top-level binding | — *(must be in class)* | yes | yes | yes | yes | yes *(module-level)* |
| block scope | yes | yes | yes | `let`/`const` yes, `var` no | yes | — *(function-scoped)* |
| shadowing in inner block | not allowed | allowed | allowed | allowed | allowed | allowed |
| hoisting | — | — | — | `var` hoists; `let`/`const` TDZ | same as JS | — |
| closure capture (read) | effectively final | direct | direct | direct | direct | direct |
| closure capture (write) | — *(effectively final)* | direct | direct | direct | direct | `nonlocal` / `global` keyword |
| class name visible from method | yes | yes | yes | yes | yes | — *(use `self.` / `cls.`)* |
| dynamic name lookup | — | — | — | — | — | yes *(`globals()`, `vars()`)* |

## Type Annotations & Inference

Two questions: **how do you spell a type in source code**, and **when does the compiler (or static checker) figure it out for you**.

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| explicit type on var | `int x = 1;` | `val x: Int = 1` | `val x: Int = 1` | — | `const x: number = 1` | `x: int = 1` |
| inferred type on var | `var x = 1;` *(10+)* | `val x = 1` | `val x = 1` | `const x = 1` | `const x = 1` | `x = 1` |
| function param type | `int x` | `x: Int` | `x: Int` | — | `x: number` | `x: int` |
| function return type | `int foo()` | `def foo(): Int` | `fun foo(): Int` | — | `function foo(): number` | `def foo() -> int:` |
| type alias | — | `type Foo = Int` | `typealias Foo = Int` | — | `type Foo = number` | `Foo = int` or `type Foo = int` *(3.12+)* |
| union type | — | `Int \| String` *(Scala 3)* | — *(sealed hierarchy)* | — | `number \| string` | `int \| str` *(3.10+)* |
| intersection type | — | `A & B` *(Scala 3)* | — | — | `A & B` | — |
| literal type | — | yes *(Scala 3)* | — | — | `"on" \| "off"` | `Literal["on", "off"]` |
| nullable in the type | — *(use `Optional`)* | — *(use `Option[T]`)* | `T?` | runtime *(no types)* | `T \| null` | `T \| None` |
| when types are checked | compile | compile | compile | runtime *(none)* | compile *(erased at runtime)* | static-only *(mypy, pyright)* |

## Generics

A single declaration parameterized over types. The big questions: **how to declare**, **how to bound**, and **how subtyping relates to the type parameter** (variance).

![Variance — covariant, contravariant, invariant](https://raw.githubusercontent.com/schemabotview/polyglot/main/img/variance.svg)

- **Covariance (`out T` / `+T`)** — `T` only flows *out* (return positions). Subtyping is preserved: `List<Dog>` is a `List<Animal>` because every read returns a `Dog`, which is also an `Animal`. Read-only collections fit here.
- **Contravariance (`in T` / `-T`)** — `T` only flows *in* (parameter positions). Subtyping is reversed: `Consumer<Animal>` is a `Consumer<Dog>` because a consumer that handles any animal certainly handles a dog. Write-only sinks.
- **Invariance (default in Java / TypeScript)** — `T` appears both as input and output, so neither subtype direction is safe. `Box<Dog>` and `Box<Animal>` are unrelated.

Scala and Kotlin let you declare variance **at the class definition** (`+T`/`-T` in Scala, `out T`/`in T` in Kotlin). Java has only **use-site** variance via wildcards: `List<? extends Animal>` (covariant read) and `List<? super Dog>` (contravariant write). Mnemonic: **PECS — Producer Extends, Consumer Super.**

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| generic class | `class Box<T> {}` | `class Box[T]` | `class Box<T>` | — | `class Box<T> {}` | `class Box(Generic[T]):` or `class Box[T]:` *(3.12+)* |
| generic function | `<T> T foo(T x)` | `def foo[T](x: T)` | `fun <T> foo(x: T)` | — | `function foo<T>(x: T)` | `def foo[T](x: T):` *(3.12+)* |
| upper bound | `<T extends Number>` | `[T <: Number]` | `<T : Number>` | — | `<T extends number>` | `TypeVar('T', bound=int)` |
| lower bound | — *(use-site only)* | `[T >: Animal]` | — | — | — | — |
| covariance (decl-site) | — | `class Box[+T]` | `class Box<out T>` | — | — | — |
| contravariance (decl-site) | — | `class Box[-T]` | `class Box<in T>` | — | — | — |
| use-site covariance | `List<? extends T>` | `List[_ <: T]` | `List<out T>` | — | — | — |
| use-site contravariance | `List<? super T>` | `List[_ >: T]` | `List<in T>` | — | — | — |
| runtime type info | erased | erased | erased *(`inline reified T` retains)* | n/a | erased | erased *(hints only)* |

## Modules & Imports

How files become a namespace and how names cross file boundaries.

| Sub-feature | Java | Scala | Kotlin | JavaScript | TypeScript | Python |
|---|---|---|---|---|---|---|
| namespace header | `package com.foo;` | `package com.foo` | `package com.foo` | filesystem path | filesystem path | filesystem path |
| file = compilation unit | one public class *(by convention)* | freer | freer | yes *(ESM)* | yes | yes |
| import single name | `import java.util.List;` | `import java.util.List` | `import java.util.List` | `import { List } from './m'` | `import { List } from './m'` | `from m import List` |
| import all from namespace | `import java.util.*;` | `import java.util.*` | `import java.util.*` | `import * as m from './m'` | `import * as m from './m'` | `from m import *` *(discouraged)* |
| import with alias | — | `import java.util.{List => L}` | `import java.util.List as L` | `import { List as L } from './m'` | same as JS | `from m import List as L` |
| relative import | — *(absolute only)* | — | — | `./m`, `../m` | same as JS | `from .m import X` |
| default visibility | package-private | public | public | not exported | not exported | public *(`_` prefix = convention)* |
| nested namespace | sub-package = sub-folder | sub-package = sub-folder | sub-package = sub-folder | nested folders | nested folders | sub-package = sub-folder *(`__init__.py`)* |

## Notes — when a cell isn't enough

**Java's "must be in a class".** Every top-level binding must live inside a `class`, `record`, `enum`, or `interface`. No free-floating functions or variables at file level. This is why "hello world" requires `public static void main`, and why scripting feels awkward in Java compared to Scala/Kotlin/Python.

**JavaScript hoisting and TDZ.** A `var x` declaration is hoisted to the top of the enclosing *function* and initialized to `undefined`. Reading `x` before its declaration line returns `undefined` instead of an error. `let` and `const` also "exist" from block start, but reading them before the declaration line throws a `ReferenceError` — this window is the **Temporal Dead Zone (TDZ)**. Modern advice: always `let`/`const`, never `var`.

**Python `nonlocal` and `global`.** Closures can *read* enclosing names freely. *Writing* requires an explicit declaration: `nonlocal x` to rebind in an enclosing function, `global x` to rebind a module-level name. Without these, the assignment silently creates a new local that shadows the outer name — a common confusing bug.

**Effectively final in Java.** A lambda or anonymous class can capture local variables only if they're *effectively final* — never reassigned after initialization. The Java compiler enforces this to avoid the JS-style "all closures share one mutable variable" footgun (classic `for (var i = 0; ...) setTimeout(() => console.log(i))`).

**Type erasure across the board.** Java, Scala, Kotlin all erase generic type parameters at runtime — `List<String>` is just `List` in bytecode. TypeScript strips all types at compile time; the runtime is plain JavaScript. Python type hints exist at runtime (in `__annotations__`) but the interpreter never enforces them. **Kotlin's `inline reified T`** is the lone escape hatch: at the call site, the compiler inlines the function body and substitutes the real type, so `T::class` works inside.

**Scala 3 vs Scala 2 type system.** Union types (`A | B`), intersection types (`A & B`), literal types (`"on"`), and many more were added in Scala 3. Scala 2 modeled the same patterns with sealed traits and ADT encoding — heavier but functional. Kotlin chose sealed hierarchies over union types and has stayed there.

**TypeScript `type` vs `interface`.** Both create named types. `interface` is *open* (can be re-opened in another file to add members), `type` is *closed*. Use `interface` for extensible object shapes, `type` for unions/intersections/tuples/mapped types. Mostly stylistic — no semantic gap for plain object types.

**Python type hints are runtime-optional.** `def foo(x: int) -> str:` looks like Java but isn't enforced — Python won't reject `foo("hello")`. Enforcement is the job of mypy/pyright/pytype as a separate static-check step. This is gradual typing, by design, and the most common surprise for Java/TypeScript programmers.

**PECS — Producer Extends, Consumer Super** (Java idiom). When using `List<? extends T>`, you can read `T` *out* (producer). When using `List<? super T>`, you can write `T` *in* (consumer). Memorize the mnemonic — even seasoned Java programmers re-derive it from scratch every six months otherwise.